# 07 — Deep Learning Anomaly Detection

## 1. Objective

This notebook develops and examines deep-learning models for unsupervised anomaly detection in the Secure Water Treatment (SWaT) dataset.

The analysis builds on the data preparation, chronological partitioning, and LSTM Autoencoder work completed in the preceding notebooks. It focuses on two additional architectures: a Temporal Convolutional Network Autoencoder (TCN Autoencoder) and Unsupervised Anomaly Detection on Multivariate Time Series (USAD).

The objective is to investigate whether these architectures can learn patterns from normal industrial process observations and assign higher anomaly scores to observations that differ from those learned patterns.

## 2. Scope

This notebook covers:

- TCN Autoencoder methodology, implementation, training, and reconstruction-score analysis.
- USAD methodology, implementation, training, and anomaly-score analysis.
- Documentation of model configurations, training behaviour, computational requirements, and observed limitations.
- Preservation and verification of the resulting model artifacts and relevant outputs.

The LSTM Autoencoder developed in Notebook 05 remains part of the project's deep-learning model set. Its existing implementation and artifacts will be reused where appropriate rather than unnecessarily reproduced.

## 3. Methodological Principles

All implementations must follow the project's established data and evaluation protocol.

- Train the models using the designated normal training partition.
- Preserve chronological separation between training, threshold-calibration, and validation observations.
- Reuse the established 42-feature definition and training-fitted scaler where appropriate.
- Do not use attack labels to fit models, select model configurations, or determine anomaly thresholds.
- Record actual training outputs, score distributions, and computational costs without inventing results.
- Document observations and limitations before proceeding to the next stage.
- Keep final threshold selection and formal evaluation in Notebook 08.

## 4. Expected Outcome

The notebook will provide reproducible implementations and documented results for the TCN Autoencoder and USAD. Together with the existing LSTM Autoencoder, these models will form the project's deep-learning detector set for subsequent threshold calibration, comparative evaluation, and error analysis.

## 5. TCN Autoencoder — Methodological Foundations

### 5.1 Objective

The Temporal Convolutional Network Autoencoder (TCN Autoencoder) is an unsupervised deep-learning model that learns to reconstruct multivariate time-series windows from normal industrial process observations.

Unlike the LSTM Autoencoder, which uses recurrent layers to represent temporal dependencies, the TCN Autoencoder uses temporal convolutions. Dilated convolutions allow the network to incorporate information from a wider temporal context without requiring a recurrent operation at every timestep.

The model will be trained using the designated normal training partition. Its reconstruction errors will provide anomaly-oriented scores for subsequent analysis.

### 5.2 Input Representation

Each model input is a multivariate time-series window:

$$
X \in \mathbb{R}^{W \times F}
$$

where:

- $W = 30$ is the established sequence-window length.
- $F = 42$ is the number of selected sensor and actuator features.

A batch of windows is represented as:

$$
X \in \mathbb{R}^{B \times W \times F}
$$

where $B$ is the batch size.

The implementation must account for PyTorch's expected input layout for one-dimensional convolutions, which is typically:

$$
X \in \mathbb{R}^{B \times F \times W}
$$

Any necessary dimension permutation must be explicit and verified.

### 5.3 Temporal Convolutions and Dilation

A temporal convolution applies learnable filters across the time dimension. Dilation introduces gaps between filter positions, allowing a convolutional layer to access a wider temporal range without proportionally increasing the kernel size.

For a one-dimensional dilated causal convolution, the operation is:

$$
y_t = \sum_{i=0}^{K-1} w_i \, x_{t - d \cdot i}
$$

where:

- $x_t$ is the input sequence.
- $y_t$ is the output at timestep $t$.
- $w_i$ represents a learnable filter coefficient.
- $K$ is the kernel size.
- $d$ is the dilation factor.

The convolution is causal when the input is padded with $(K-1) \cdot d$ zeros on its left edge and the padded sequence is indexed from $1$; equivalently, $x_p = 0$ for all $p \leq 0$. Under that padding, the output $y_t$ depends only on $x_t$ and the preceding positions $x_{t-d}, x_{t-2d}, \ldots, x_{t-(K-1)d}$. No future observation $x_{t+j}$ with $j > 0$ contributes to $y_t$.

### 5.4 Receptive Field

The receptive field is the number of input timesteps that can influence an output position.

For a stack of $L$ dilated convolutional layers with common kernel size $K$ and dilations $d_1, d_2, \ldots, d_L$, the receptive field is:

$$
R = 1 + (K-1) \sum_{\ell=1}^{L} d_\ell
$$

$L$ in this formula is the total number of convolutional layers, not the number of TCN blocks. If a block contains two convolutions with the same dilation factor, its contribution to the sum is $2 \cdot d$ and to $L$ is $2$. This distinction matters when comparing against published configurations, which typically stack 2 convolutions per block.

For the reference configuration with four TCN blocks, dilation schedule $1, 2, 4, 8$, and kernel size $3$:

- If each block contains 1 convolution ($L = 4$, $d = [1, 2, 4, 8]$): $R = 1 + 2 \cdot (1+2+4+8) = 31$.
- If each block contains 2 convolutions ($L = 8$, $d = [1,1,2,2,4,4,8,8]$): $R = 1 + 2 \cdot 30 = 61$.
- If each block contains 3 convolutions ($L = 12$, $d = [1,1,1,2,2,2,4,4,4,8,8,8]$): $R = 1 + 2 \cdot 45 = 91$.

The reference reports a receptive field of 90 timesteps. That value is consistent with 3 convolutions per block (giving 91) but not with 1 or 2; it must be verified against the actual implementation, not inferred from the dilation schedule alone. The count reported here assumes no stride, no downsampling between blocks, and no extra conv layers on the residual path — any of those would change the effective receptive field.

Because this project's established input window contains 30 timesteps, the implementation must also be checked to confirm how its receptive field compares with the window length. If the computed receptive field exceeds 30, the network's output at early window positions will be computed from padded input only, and its reconstruction of those positions may be less informative than the middle and end of the window.

### 5.5 Encoder and Decoder

The encoder transforms the input sequence into a more compact representation of its temporal patterns. Dilated convolutional layers allow the encoder to represent dependencies across multiple temporal scales.

The decoder maps the encoded representation back to the original sequence dimensions.

For an input window $X$, the reconstructed output is denoted by:

$$
\hat{X} = f_{\theta}(X)
$$

where $f_{\theta}$ represents the trainable encoder–decoder network.

The reconstructed output must preserve the input window's temporal and feature dimensions:

$$
\hat{X} \in \mathbb{R}^{B \times W \times F}
$$

This shape requirement will be verified before training.

### 5.6 Reconstruction Loss

The model uses mean squared error (MSE) as its reconstruction objective:

$$
\mathcal{L}_{\mathrm{MSE}}
=
\frac{1}{B W F}
\sum_{b=1}^{B}
\sum_{t=1}^{W}
\sum_{f=1}^{F}
\left( X_{b,t,f} - \hat{X}_{b,t,f} \right)^2
$$

where:

- $X_{b,t,f}$ is the observed feature value.
- $\hat{X}_{b,t,f}$ is its reconstructed value.
- $B$, $W$, and $F$ denote batch size, window length, and feature count.

Minimizing this loss encourages the model to reconstruct the patterns represented in its training data.

### 5.7 Anomaly Scoring

After training, reconstruction error is calculated for each evaluated sequence.

For sequence $i$, the mean squared reconstruction error is:

$$
s_i =
\frac{1}{W F}
\sum_{t=1}^{W}
\sum_{f=1}^{F}
\left( X_{i,t,f} - \hat{X}_{i,t,f} \right)^2
$$

Higher scores indicate greater reconstruction error and are interpreted as greater anomalousness under this scoring convention.

High reconstruction error does not by itself establish a physical fault or cyberattack. Formal classifications will depend on the threshold-calibration and evaluation protocol defined in Notebook 08.

### 5.8 Training and Evaluation Controls

The implementation preserves the project's existing methodological controls:

- Use the established normal training partition for model fitting.
- Reuse the existing 42-feature definition and training-fitted scaler.
- Preserve chronological separation between training, threshold-calibration, and validation data.
- Avoid using attack labels for model fitting, architecture selection, or threshold selection.
- Verify model input and output dimensions before training.
- Record the actual configuration, training history, runtime, and reconstruction-score distributions.
- Save the fitted model and verify that it can be reloaded.
- Defer final threshold selection and formal evaluation to Notebook 08.

The TCN Autoencoder will be assessed using recorded evidence from this implementation. Results from the reference notebook will not be assumed to transfer to the current dataset or configuration.